In [ ]:
# Install dependensi yang mungkin belum ada
import subprocess, sys
pkgs = ['matplotlib', 'threadpoolctl']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Semua dependensi tersedia.')


# 04 — Test Final dan Pengukuran Waktu Prediksi (Proyek TTKI P1)

Notebook ini melengkapi notebook 02–03 dengan dua hal yang diminta proposal (bagian 9):

1. **Test final** — setiap skenario (A–D) dan model (LR, RF) dilatih ulang pada **seluruh development set**,
   lalu dievaluasi **satu kali** pada test final dengan threshold 0,5. Tidak ada pemilihan parameter, fitur, atau threshold dari test final.
2. **Waktu prediksi** — waktu `predict_proba` untuk batch tetap 1.000 baris (warm-up, 30 pengulangan; median dan IQR),
   dengan perangkat dan pengaturan thread yang dicatat. Fitur dianggap sudah tersedia (bukan waktu unduh halaman/ekstraksi fitur).

**Prasyarat:** notebook 02 dan 03 sudah dijalankan (ada `split_indices_random.npz`, `split_indices_grouped.npz`, `all_scenarios_cv_summary.csv`).

**Keluaran** (di `results/`): `test_final_results.csv`, `test_final_selected_features_D.csv`, `timing_results.csv`, `timing_environment.json`, `final_table_cv_vs_test.csv`.

**Catatan penting:** Bagian 3 (test final) sengaja dijaga dua pengaman (`RUN_FINAL_TEST` dan penolakan menimpa hasil lama).
Jalankan **sekali**, setelah rancangan dibekukan dan disepakati dengan Dika dan dosen. Bagian 4 (waktu) tidak memakai label test, jadi boleh diulang.

In [ ]:
import os, json, time, gc, platform, hashlib, warnings
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix
from threadpoolctl import threadpool_limits, threadpool_info
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)   # penalty= pada LogisticRegression di sklearn >= 1.8

# --- lokasi berkas (mengikuti notebook 02/03; juga jalan jika semua berkas ada di satu folder) ---
CFG_PATH = next(p for p in ["../configs/experiment_config.json", "configs/experiment_config.json",
                            "experiment_config.json"] if os.path.exists(p))
cfg = json.load(open(CFG_PATH))
ROOT = os.path.dirname(os.path.dirname(CFG_PATH)) if os.path.basename(os.path.dirname(CFG_PATH) or ".") == "configs" else (os.path.dirname(CFG_PATH) or ".")
RESULTS_DIR = os.path.join(ROOT, "results") if os.path.isdir(os.path.join(ROOT, "results")) else ROOT
DATA_PATH = next(p for p in [os.path.join(ROOT, "data", cfg["dataset"]["file"]),
                             os.path.join(ROOT, cfg["dataset"]["file"]),
                             "/mnt/user-data/uploads/" + cfg["dataset"]["file"]] if os.path.exists(p))

SEED = cfg["seed"]
THR = cfg["evaluation"]["threshold"]          # 0.5, tidak boleh diubah berdasarkan test final
RUN_FINAL_TEST = False                        # ubah ke True SEKALI setelah rancangan dibekukan
OVERWRITE_FINAL = False                       # pengaman: tolak menimpa test_final_results.csv yang sudah ada
K_FEATURES = 5
MODELS = ["logistic_regression", "random_forest"]

# parameter pengukuran waktu (proposal bagian 9)
BATCH_SIZE, N_WARMUP, N_REPEAT = 1000, 3, 30

print("sklearn", sklearn.__version__, "| pandas", pd.__version__, "| seed", SEED, "| threshold", THR)
print("Config :", CFG_PATH, "\nResults:", RESULTS_DIR, "\nData   :", DATA_PATH)

## 1. Data, fitur, dan indeks split
Indeks split dibaca dari berkas yang sudah disimpan (bukan dibuat ulang), sehingga test final persis sama dengan yang dipakai CV.

In [ ]:
df = pd.read_csv(DATA_PATH)
y = (1 - df["label"]).values                       # phishing = 1
F_URL, F_PAGE = cfg["features"]["F_URL"], cfg["features"]["F_Page"]
FEATURES_A = F_URL + F_PAGE
assert not set(FEATURES_A) & set(cfg["features"]["excluded_unclear"])

# pastikan berkas data sama dengan yang tercatat saat audit
sha = hashlib.sha256(open(DATA_PATH, "rb").read()).hexdigest()
assert sha in {cfg["dataset"]["sha256"], cfg["dataset"]["sha256_lf"]}, "Checksum dataset tidak dikenal"
print("Checksum dataset terverifikasi")

splits = {"random":  np.load(os.path.join(RESULTS_DIR, "split_indices_random.npz")),
          "grouped": np.load(os.path.join(RESULTS_DIR, "split_indices_grouped.npz"))}
for name, s in splits.items():
    dev, test = s["dev"], s["test"]
    assert len(set(dev) & set(test)) == 0, f"dev dan test beririsan pada split {name}"
    print(f"{name:8s}: dev {len(dev):,} | test {len(test):,} | phishing dev/test = {y[dev].mean():.4f}/{y[test].mean():.4f}")

# skenario: (kolom fitur, protokol split, k seleksi fitur)
SCENARIOS = {"A": (FEATURES_A, "random",  None),
             "B": (F_URL,      "random",  None),
             "C": (F_URL,      "grouped", None),
             "D": (F_URL,      "grouped", K_FEATURES)}
for sc, (cols, proto, k) in SCENARIOS.items():
    print(f"Skenario {sc}: {len(cols)} fitur masuk pipeline | split {proto} | seleksi k={k}")

## 2. Pipeline dan fungsi bantu
Pipeline identik dengan notebook 02/03 (imputasi, scaling LR, `SelectKBest` untuk D — semuanya di dalam `Pipeline`).
`fit_all()` melatih tiap skenario × model **sekali** pada development set lalu menyimpannya, sehingga test final dan pengukuran waktu memakai model yang sama.

In [ ]:
def make_pipeline(model_name, k_select=None):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if k_select:
        steps.append(("select", SelectKBest(f_classif, k=k_select)))
    p = cfg["models"][model_name]
    if model_name == "logistic_regression":
        steps += [("scale", StandardScaler()),
                  ("clf", LogisticRegression(penalty=p["penalty"], C=p["C"], solver=p["solver"], max_iter=p["max_iter"]))]
    else:
        steps.append(("clf", RandomForestClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"],
                     max_features=p["max_features"], random_state=p["random_state"], n_jobs=-1)))
    return Pipeline(steps)

def evaluate(y_true, proba, thr=THR):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"f1_phishing": f1_score(y_true, pred), "roc_auc": roc_auc_score(y_true, proba),
            "precision": precision_score(y_true, pred), "recall": recall_score(y_true, pred),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

FITTED = {}      # (skenario, model) -> pipeline terlatih pada development set
FIT_SECONDS = {}

def fit_all():
    for sc, (cols, proto, k) in SCENARIOS.items():
        dev = splits[proto]["dev"]
        X_dev = df[cols].astype(float).values[dev]
        for m in MODELS:
            if (sc, m) in FITTED:
                continue
            pipe = make_pipeline(m, k)
            t0 = time.perf_counter(); pipe.fit(X_dev, y[dev]); FIT_SECONDS[(sc, m)] = time.perf_counter() - t0
            FITTED[(sc, m)] = pipe
            print(f"[{sc}] {m:20s} dilatih pada {len(dev):,} baris ({FIT_SECONDS[(sc, m)]:.1f} dtk)")

def feature_names(sc, pipe):
    cols = SCENARIOS[sc][0]
    if "select" in pipe.named_steps:
        mask = pipe.named_steps["select"].get_support()
        return [c for c, keep in zip(cols, mask) if keep]
    return list(cols)

## 3. Test final (dijaga)
Setiap kombinasi dievaluasi **satu kali**. Ini menghasilkan F1 phishing, ROC-AUC, precision, recall, dan confusion matrix per skenario per model.
Hasil dilaporkan **terpisah** dari rata-rata CV (CV: rata-rata ± simpangan baku; test final: satu angka).

In [ ]:
FINAL_PATH = os.path.join(RESULTS_DIR, "test_final_results.csv")

if not RUN_FINAL_TEST:
    print("Test final belum dijalankan (RUN_FINAL_TEST = False).")
elif os.path.exists(FINAL_PATH) and not OVERWRITE_FINAL:
    print(f"DITOLAK: {FINAL_PATH} sudah ada. Test final tidak boleh diulang untuk memilih hasil terbaik.")
    print("Jika memang perlu (mis. bug kode), setel OVERWRITE_FINAL = True dan catat alasannya di README.")
else:
    fit_all()
    rows, sel_rows = [], []
    for (sc, m), pipe in FITTED.items():
        cols, proto, k = SCENARIOS[sc]
        te = splits[proto]["test"]
        proba = pipe.predict_proba(df[cols].astype(float).values[te])[:, 1]
        met = evaluate(y[te], proba)
        feats = feature_names(sc, pipe)
        rows.append({"scenario": sc, "protocol": proto, "model": m, "n_features": len(feats),
                     "n_dev": len(splits[proto]["dev"]), "n_test": len(te),
                     "fit_seconds_dev": round(FIT_SECONDS[(sc, m)], 2), "threshold": THR, **met})
        if k:
            sel_rows += [{"scenario": sc, "model": m, "feature": f} for f in feats]
    final = pd.DataFrame(rows).sort_values(["scenario", "model"]).reset_index(drop=True)
    final.to_csv(FINAL_PATH, index=False)
    pd.DataFrame(sel_rows).to_csv(os.path.join(RESULTS_DIR, "test_final_selected_features_D.csv"), index=False)

    # catat bahwa test final sudah disentuh
    cfg["evaluation"]["test_final_touched"] = True
    json.dump(cfg, open(CFG_PATH, "w"), indent=2, ensure_ascii=False)

    show = ["scenario", "model", "n_features", "f1_phishing", "roc_auc", "precision", "recall", "tn", "fp", "fn", "tp"]
    print(final[show].round(4).to_string(index=False))
    print("\nFitur terpilih skenario D pada development set penuh:")
    print(pd.DataFrame(sel_rows).groupby("model")["feature"].apply(lambda s: ", ".join(s)).to_string())

## 4. Waktu prediksi
- **Yang diukur:** `pipeline.predict_proba` (imputasi + scaling/seleksi + model) pada batch tetap 1.000 baris yang fiturnya sudah tersedia.
  Bukan waktu pengunduhan halaman atau ekstraksi fitur, sehingga angka ini **tidak** mewakili seluruh biaya deteksi.
- **Batch:** 1.000 baris diambil acak (seed 42) dari **development set** dan dipakai sama untuk semua model dan skenario dengan protokol split yang sama. Label tidak dipakai, jadi test final tidak tersentuh.
- **Prosedur:** `gc.collect()`, 3 pemanggilan warm-up (tidak dihitung), lalu 30 pengulangan dengan `time.perf_counter`.
- **Dua mode thread:** *default* (RF memakai `n_jobs=-1` sesuai konfigurasi, LR mengikuti BLAS) dan *1 thread* (`n_jobs=1` dan `threadpool_limits(1)`),
  karena pada batch sekecil 1.000 baris overhead paralelisme RF dapat lebih besar daripada komputasinya.
- **Dilaporkan:** median dan rentang antarkuartil (IQR = Q3 − Q1) dalam milidetik, serta perangkat dan pengaturan thread.

In [ ]:
fit_all()

def time_predict(pipe, X_batch, n_warmup=N_WARMUP, n_repeat=N_REPEAT):
    for _ in range(n_warmup):
        pipe.predict_proba(X_batch)
    gc.collect()
    ts = np.empty(n_repeat)
    for i in range(n_repeat):
        t0 = time.perf_counter(); pipe.predict_proba(X_batch); ts[i] = (time.perf_counter() - t0) * 1000
    q1, med, q3 = np.percentile(ts, [25, 50, 75])
    return {"median_ms": med, "q1_ms": q1, "q3_ms": q3, "iqr_ms": q3 - q1, "min_ms": ts.min(), "max_ms": ts.max()}

rng = np.random.RandomState(SEED)
batch_by_proto = {proto: rng.choice(splits[proto]["dev"], size=BATCH_SIZE, replace=False) for proto in ("random", "grouped")}
np.savez(os.path.join(RESULTS_DIR, "timing_batch_indices_paired.npz"), **batch_by_proto)
timing_rows = []
for sc, (cols, proto, k) in SCENARIOS.items():
    batch_idx = batch_by_proto[proto]
    X_batch = df[cols].astype(float).values[batch_idx]
    for m in MODELS:
        pipe = FITTED[(sc, m)]
        for mode in ["default", "1_thread"]:
            if mode == "1_thread":
                if m == "random_forest":
                    pipe.set_params(clf__n_jobs=1)
                with threadpool_limits(limits=1):
                    r = time_predict(pipe, X_batch)
                if m == "random_forest":
                    pipe.set_params(clf__n_jobs=-1)      # kembalikan ke konfigurasi awal
            else:
                r = time_predict(pipe, X_batch)
            timing_rows.append({"scenario": sc, "model": m, "thread_mode": mode,
                                "n_features": len(feature_names(sc, pipe)), "batch_size": BATCH_SIZE,
                                "n_warmup": N_WARMUP, "n_repeat": N_REPEAT, **r})
            print(f"[{sc}] {m:20s} {mode:8s} median {r['median_ms']:8.2f} ms  IQR {r['iqr_ms']:6.2f} ms")

timing = pd.DataFrame(timing_rows).round(3)
timing.to_csv(os.path.join(RESULTS_DIR, "timing_results_paired.csv"), index=False)

env = {"python": platform.python_version(), "platform": platform.platform(), "processor": platform.processor(),
       "cpu_count_logical": os.cpu_count(), "sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "threadpools": [{k: v for k, v in t.items() if k in ("internal_api", "num_threads", "version")} for t in threadpool_info()],
       "batch_size": BATCH_SIZE, "n_warmup": N_WARMUP, "n_repeat": N_REPEAT,
       "catatan": "Waktu predict_proba dari fitur yang sudah tersedia; tidak termasuk pengunduhan halaman/ekstraksi fitur."}
json.dump(env, open(os.path.join(RESULTS_DIR, "timing_environment_paired.json"), "w"), indent=2)
print(); print(json.dumps(env, indent=2))

In [ ]:
# ringkasan: median [Q1-Q3] ms per skenario x model x mode, plus rasio D terhadap C (pengaruh pengurangan fitur, H2)
pv = timing.assign(txt=lambda d: d.median_ms.map("{:.2f}".format) + " [" + d.q1_ms.map("{:.2f}".format) + "-" + d.q3_ms.map("{:.2f}".format) + "]")
print(pv.pivot_table(index=["model", "thread_mode"], columns="scenario", values="txt", aggfunc="first").to_string())
print()
for m in MODELS:
    for mode in ["default", "1_thread"]:
        g = timing[(timing.model == m) & (timing.thread_mode == mode)].set_index("scenario")["median_ms"]
        print(f"{m:20s} {mode:8s} D/C = {g['D'] / g['C']:.2f}  (A/B = {g['A'] / g['B']:.2f})")

## 5. Tabel gabungan CV vs test final (untuk paper)
CV = rata-rata ± simpangan baku 5 fold pada development set; Test = satu evaluasi pada test final.
Selisih CV–test yang kecil menunjukkan estimasi CV konsisten dengan test, tetapi **tidak** membuktikan ketahanan terhadap kampanye phishing baru.

In [ ]:
if os.path.exists(FINAL_PATH):
    final = pd.read_csv(FINAL_PATH)
    mf = pd.read_csv(os.path.join(RESULTS_DIR, "metrics_per_fold.csv"))
    cv = mf.groupby(["scenario", "model"])[["f1_phishing", "roc_auc"]].agg(["mean", "std"])
    cv.columns = [f"cv_{a}_{b}" for a, b in cv.columns]
    tab = final.set_index(["scenario", "model"]).join(cv).reset_index()
    tab["F1 CV"]  = tab.apply(lambda r: f"{r.cv_f1_phishing_mean:.4f} ± {r.cv_f1_phishing_std:.4f}", axis=1)
    tab["F1 test"] = tab.f1_phishing.map("{:.4f}".format)
    tab["AUC CV"]  = tab.apply(lambda r: f"{r.cv_roc_auc_mean:.4f} ± {r.cv_roc_auc_std:.4f}", axis=1)
    tab["AUC test"] = tab.roc_auc.map("{:.4f}".format)
    tab["Selisih F1 (test - CV)"] = (tab.f1_phishing - tab.cv_f1_phishing_mean).map("{:+.4f}".format)
    out = tab[["scenario", "model", "n_features", "F1 CV", "F1 test", "Selisih F1 (test - CV)", "AUC CV", "AUC test",
               "precision", "recall"]].round(4)
    out.to_csv(os.path.join(RESULTS_DIR, "final_table_cv_vs_test.csv"), index=False)
    print(out.to_string(index=False))
else:
    print("Belum ada test_final_results.csv; jalankan bagian 3 dengan RUN_FINAL_TEST = True terlebih dahulu.")

## 6. Visualisasi hasil (H1, H2, H3)
Plot 4 panel yang membandingkan F1 CV vs test final antar skenario, plus waktu prediksi.
Disimpan ke `results/plots_h1h2h3_paired.png`. Jalankan setelah bagian 3 (test final) sudah selesai.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# --- baca data dari CSV yang sudah ada ---
cv_summ = pd.read_csv(os.path.join(RESULTS_DIR, "all_scenarios_cv_summary.csv"), header=[0,1], index_col=[0,1])
timing_df = pd.read_csv(os.path.join(RESULTS_DIR, "timing_results_paired.csv"))

if not os.path.exists(FINAL_PATH):
    print("Belum ada test_final_results.csv — jalankan bagian 3 dulu (RUN_FINAL_TEST = True).")
else:
    final_df = pd.read_csv(FINAL_PATH)

    # --- bentuk dict agar mudah diakses ---
    def cv_val(sc, model, metric):
        m = "logistic_regression" if model == "LR" else "random_forest"
        return float(cv_summ.loc[(sc, m), (metric, "mean")]), float(cv_summ.loc[(sc, m), (metric, "std")])

    def test_val(sc, model):
        m = "logistic_regression" if model == "LR" else "random_forest"
        row = final_df[(final_df.scenario == sc) & (final_df.model == m)].iloc[0]
        return row.f1_phishing, row.roc_auc, row.precision, row.recall

    def t_med(sc, model, mode="default"):
        m = "logistic_regression" if model == "LR" else "random_forest"
        row = timing_df[(timing_df.scenario == sc) & (timing_df.model == m) & (timing_df.thread_mode == mode)].iloc[0]
        return row.median_ms, row.iqr_ms

    CLR, CRF, CGRAY = '#2563EB', '#DC2626', '#6B7280'
    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)
    w = 0.18

    def draw_panel(ax, scenarios, title, ylim, model_list=["LR","RF"]):
        x = np.arange(len(scenarios))
        for i, (m, c) in enumerate(zip(model_list, [CLR, CRF])):
            f1_cv  = [cv_val(s, m, "f1_phishing")[0] for s in scenarios]
            f1_std = [cv_val(s, m, "f1_phishing")[1] for s in scenarios]
            f1_t   = [test_val(s, m)[0] for s in scenarios]
            off = (i - 0.5) * w * 2
            ax.bar(x + off - w/2, f1_cv, w, color=c, alpha=0.65)
            ax.bar(x + off + w/2, f1_t,  w, color=c, alpha=1.0, edgecolor='white', linewidth=0.5)
            ax.errorbar(x + off - w/2, f1_cv, yerr=f1_std, fmt='none', color='black', capsize=3, linewidth=1)
        ax.set_xticks(x); ax.set_ylim(*ylim)
        ax.set_ylabel("F1-score (kelas phishing)")
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"{v:.3f}"))
        if len(scenarios) == 2:
            ax.axvline(0.5, color=CGRAY, linewidth=0.8, linestyle='--', alpha=0.5)

    # Panel 1 — H3: A vs B
    ax1 = fig.add_subplot(gs[0, 0])
    draw_panel(ax1, ['A','B'], 'H3 — Pengaruh kelompok fitur\n(A→B: buang fitur halaman)', (0.994, 1.001))
    ax1.set_xticklabels(['A\n(URL+Halaman)', 'B\n(URL saja)'])
    for m, c in [('LR', CLR), ('RF', CRF)]:
        d = test_val('B', m)[0] - test_val('A', m)[0]
        ax1.text(0.5, test_val('B',m)[0]-0.0002, f'ΔF1={d:+.4f}', ha='center', fontsize=7.5, color=c)

    # Panel 2 — H1: B vs C
    ax2 = fig.add_subplot(gs[0, 1])
    draw_panel(ax2, ['B','C'], 'H1 — Sensitivitas protokol split\n(B→C: registered-domain-disjoint)', (0.994, 1.001))
    ax2.set_xticklabels(['B\n(split acak)', 'C\n(split domain)'])
    for m, c in [('LR', CLR), ('RF', CRF)]:
        d = test_val('C', m)[0] - test_val('B', m)[0]
        ax2.text(0.5, test_val('C',m)[0]-0.0002, f'ΔF1={d:+.4f}', ha='center', fontsize=7.5, color=c)
    ax2.text(0.5, 1.0005, 'ΔF1 kecil pada split ini;\nbukan bukti bebas leakage',
             ha='center', fontsize=8, color='#059669', style='italic')

    # Panel 3 — H2 performa: C vs D
    ax3 = fig.add_subplot(gs[1, 0])
    draw_panel(ax3, ['C','D'], 'H2 — Pengaruh pengurangan fitur (performa)\n(C→D: 18 fitur → 5 fitur)', (0.980, 1.001))
    ax3.set_xticklabels(['C\n(18 fitur)', 'D\n(5 fitur)'])
    for m, c in [('LR', CLR), ('RF', CRF)]:
        dt = test_val('D',m)[0] - test_val('C',m)[0]
        dc = cv_val('D',m,'f1_phishing')[0] - cv_val('C',m,'f1_phishing')[0]
        ax3.text(0.5, test_val('D',m)[0]-0.0012,
                 f'ΔF1={dt:+.4f} (test)\n{dc:+.4f} (CV)', ha='center', fontsize=7, color=c)

    # Panel 4 — H2 waktu: B, C, D
    ax4 = fig.add_subplot(gs[1, 1])
    sc_t = ['B','C','D']; x4 = np.arange(3); w4 = 0.3
    for i, (m, c) in enumerate([('LR', CLR), ('RF', CRF)]):
        meds = [t_med(s, m)[0] for s in sc_t]
        model_key = "logistic_regression" if m == "LR" else "random_forest"
        q = timing_df[(timing_df.model == model_key) & (timing_df.thread_mode == "default")].set_index("scenario").loc[sc_t]
        errors = np.array([q.median_ms - q.q1_ms, q.q3_ms - q.median_ms])
        off = (i - 0.5) * w4
        ax4.bar(x4 + off, meds, w4, color=c, alpha=0.85, label=m)
        ax4.errorbar(x4 + off, meds, yerr=errors, fmt='none', color='black', capsize=4, linewidth=1.2)
    ax4.set_xticks(x4)
    ax4.set_xticklabels(['B\n(18 fitur, acak)', 'C\n(18 fitur, domain)', 'D\n(5 fitur, seleksi)'])
    ax4.set_ylabel('Waktu prediksi (ms, median, Q1–Q3)'); ax4.legend(fontsize=9)
    ax4.set_title('H2 — Pengaruh pengurangan fitur (waktu)\nmode default, batch 1.000 baris', fontsize=10, fontweight='bold')
    for m, c in [('LR', CLR), ('RF', CRF)]:
        ratio = t_med('D', m)[0] / t_med('C', m)[0]
        label = '↑lebih lambat' if ratio > 1 else '↓lebih cepat'
        ax4.text(x4[2] + (-0.15 if m=='LR' else 0.15), t_med('D',m)[0]+2,
                 f'D/C={ratio:.2f}\n({label})', ha='center', fontsize=7.5, color=c)

    # Legenda global
    patches = [mpatches.Patch(facecolor=c, alpha=a, label=lbl) for c, a, lbl in [
        (CLR,0.65,'LR — CV (mean±SD)'), (CLR,1.0,'LR — Test final'),
        (CRF,0.65,'RF — CV (mean±SD)'), (CRF,1.0,'RF — Test final')]]
    fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=8.5,
               bbox_to_anchor=(0.5,-0.01), frameon=True)
    fig.suptitle('Hasil Eksperimen — Deteksi Phishing Berbasis Fitur URL\n(PhiUSIIL, n=235.795, threshold=0,5)',
                 fontsize=12, fontweight='bold', y=1.01)

    out_plot = os.path.join(RESULTS_DIR, "plots_h1h2h3_paired.png")
    plt.savefig(out_plot, dpi=180, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Plot disimpan ke: {out_plot}")

## 7. Tabel akhir untuk paper
Tabel gabungan CV + test final + waktu prediksi. Disimpan ke `results/final_table_cv_vs_test.csv` (sudah dari bagian 5) dan dicetak di sini dalam format yang siap disalin ke paper.

In [ ]:
if os.path.exists(FINAL_PATH):
    tab = pd.read_csv(os.path.join(RESULTS_DIR, "final_table_cv_vs_test.csv"))

    # format ulang untuk tampilan
    def fmt(row):
        return pd.Series({
            'Sk': row.scenario,
            'Model': 'LR' if 'logistic' in row.model else 'RF',
            'Fitur': int(row.n_features),
            'Split': 'acak' if row.scenario in ['A','B'] else 'domain',
            'F1-CV': f"{row['F1 CV']}",
            'F1-test': row['F1 test'],
            'ΔF1': row['Selisih F1 (test - CV)'],
            'AUC-CV': f"{row['AUC CV']}",
            'AUC-test': row['AUC test'],
            'Precision': f"{row.precision:.4f}",
            'Recall': f"{row.recall:.4f}",
        })

    display_tab = tab.apply(fmt, axis=1)
    print(display_tab.to_string(index=False))
    print()

    # ringkasan hipotesis
    f = final_df.set_index(['scenario','model'])
    get = lambda sc, m, col: f.loc[(sc, m), col]
    ML = 'logistic_regression'; MR = 'random_forest'
    print("=" * 60)
    print("INTERPRETASI HIPOTESIS")
    print("=" * 60)
    h1_lr = get('C',ML,'f1_phishing') - get('B',ML,'f1_phishing')
    h1_rf = get('C',MR,'f1_phishing') - get('B',MR,'f1_phishing')
    h2p_lr = get('D',ML,'f1_phishing') - get('C',ML,'f1_phishing')
    h2p_rf = get('D',MR,'f1_phishing') - get('C',MR,'f1_phishing')
    h3_lr = get('B',ML,'f1_phishing') - get('A',ML,'f1_phishing')
    h3_rf = get('B',MR,'f1_phishing') - get('A',MR,'f1_phishing')

    td = timing_df[timing_df.thread_mode=='default'].set_index(['scenario','model'])
    get_t = lambda sc, m: td.loc[(sc, m), 'median_ms']
    ML2 = 'logistic_regression'; MR2 = 'random_forest'
    h2t_lr = get_t('D',ML2) / get_t('C',ML2)
    h2t_rf = get_t('D',MR2) / get_t('C',MR2)

    print(f"H1 (B→C, protokol split)  : LR {h1_lr:+.4f} | RF {h1_rf:+.4f} → {'TIDAK terdukung' if abs(h1_lr)<0.002 else 'Terdukung'}")
    print(f"H2 (C→D, kurangi fitur)   : F1 LR {h2p_lr:+.4f} | RF {h2p_rf:+.4f}")
    print(f"                            Waktu LR D/C={h2t_lr:.2f}× | RF D/C={h2t_rf:.2f}× ({'LEBIH LAMBAT' if h2t_lr>1 else 'lebih cepat'})")
    status_h2 = 'Sebagian (performa turun, waktu tidak hemat)' if h2p_lr < -0.001 and h2t_lr > 1 else 'Terdukung'
    print(f"                            → {status_h2}")
    print(f"H3 (A→B, kelompok fitur)  : LR {h3_lr:+.4f} | RF {h3_rf:+.4f} → {'Terdukung' if abs(h3_lr)>0.002 else 'Lemah'}")
else:
    print("Belum ada test_final_results.csv — jalankan bagian 3 dulu.")